# Phase 12 — Real-Time Pipeline Throughput (GPU)

Runs **YOLOv8n detection + OSNet-AIN-x1.0 embedding** on a T4 GPU and measures effective FPS.

**Before running:** Runtime → Change runtime type → T4 GPU

In [1]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("✗ No GPU — switch runtime to T4 GPU first!")


✓ GPU: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ── 2. Install dependencies ────────────────────────────────────────────────────
import subprocess, sys
pkgs = ["ultralytics", "torchreid @ git+https://github.com/KaiyangZhou/deep-person-reid.git"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("✓ Packages installed")


✓ Packages installed


In [3]:
# ── 3. Generate synthetic sim_cam1.mp4 (300 frames, 1280×720, 25 fps) ─────────
# Simulates the coloured-block video the edge client normally processes.
# No upload needed — the timing benchmark only cares about frame resolution.
import cv2, numpy as np
from pathlib import Path

VIDEO_PATH = Path("sim_cam1.mp4")
W, H, FPS, N = 1280, 720, 25, 350
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_PATH), fourcc, FPS, (W, H))
rng = np.random.default_rng(42)
for i in range(N):
    frame = rng.integers(0, 255, (H, W, 3), dtype=np.uint8)
    writer.write(frame)
writer.release()
print(f"✓ {VIDEO_PATH} written ({N} frames, {W}×{H}, {FPS} fps)")


✓ sim_cam1.mp4 written (350 frames, 1280×720, 25 fps)


In [4]:
# ── 4. Load models ─────────────────────────────────────────────────────────────
import torch, torchreid
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"

detector = YOLO("yolov8n.pt")

# Load OSNet-AIN (pretrained weights are fine — we're measuring inference speed)
extractor = torchreid.utils.FeatureExtractor(
    model_name="osnet_ain_x1_0",
    image_size=(256, 256),
    pixel_mean=[0.485, 0.456, 0.406],
    pixel_std=[0.229, 0.224, 0.225],
    device=device,
)
print(f"✓ Models loaded on {device}")

# GPU warm-up (important for accurate timing)
dummy_frame = np.zeros((720, 1280, 3), dtype=np.uint8)
dummy_crop  = np.zeros((128, 128, 3), dtype=np.uint8)
for _ in range(3):
    detector(dummy_frame, verbose=False)
    with torch.no_grad():
        extractor([cv2.cvtColor(dummy_crop, cv2.COLOR_BGR2RGB)])
if device == "cuda":
    torch.cuda.synchronize()
print("✓ Warm-up complete")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Downloading...
From: https://drive.google.com/uc?id=1-CaioD9NaqbHK_kzSMW8VE4_3KcsRjEo
To: /root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:01<00:00, 9.67MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_ain_x1_0
- params: 2,193,616
- flops: 1,957,461,904
✓ Models loaded on cuda
✓ Warm-up complete


In [5]:
# ── 5. Run timing benchmark (mirrors phase_realtime in evaluate_system.py) ─────
import time, cv2, numpy as np, torch
from pathlib import Path

VIDEO_FPS    = 25
FRAME_BUDGET = 1000.0 / VIDEO_FPS   # 40 ms
EMBED_BUDGET = FRAME_BUDGET * 5     # 200 ms
N_FRAMES     = 300

cap = cv2.VideoCapture(str(VIDEO_PATH))
detect_times, embed_times, total_times = [], [], []
frames_read = crops_found = 0

while frames_read < N_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    frames_read += 1

    t_frame = time.perf_counter()

    # --- Detection ---
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    results = detector(frame, verbose=False, classes=[0, 1, 2, 3])
    if device == "cuda":
        torch.cuda.synchronize()
    detect_times.append((time.perf_counter() - t0) * 1000)

    # --- Embedding (best detected crop) ---
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        best_box = boxes[int(boxes.conf.argmax())]
        x1,y1,x2,y2 = (int(v) for v in best_box.xyxy[0].tolist())
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0 and min(crop.shape[:2]) >= 8:
            crops_found += 1
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            if device == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                extractor([rgb])
            if device == "cuda":
                torch.cuda.synchronize()
            embed_times.append((time.perf_counter() - t0) * 1000)

    total_times.append((time.perf_counter() - t_frame) * 1000)

cap.release()

mean_det = float(np.mean(detect_times))
p95_det  = float(np.percentile(detect_times, 95))
mean_emb = float(np.mean(embed_times))  if embed_times else 0
p95_emb  = float(np.percentile(embed_times, 95)) if embed_times else 0
mean_tot = float(np.mean(total_times))
eff_fps  = 1000.0 / mean_tot

pass_det  = mean_det < FRAME_BUDGET
pass_emb  = mean_emb < EMBED_BUDGET
pass_fps  = eff_fps  >= VIDEO_FPS * 0.8

print(f"\n{'='*58}")
print(f"  Phase 12 — Real-Time Pipeline Throughput  [{device.upper()}]")
print(f"{'='*58}")
print(f"  Frames benchmarked : {frames_read}  ({crops_found} with detection)")
print(f"  Detection  mean    : {mean_det:.1f} ms   p95: {p95_det:.1f} ms   budget: {FRAME_BUDGET:.0f} ms  {'PASS ✓' if pass_det else 'FAIL ✗'}")
print(f"  Embedding  mean    : {mean_emb:.1f} ms   p95: {p95_emb:.1f} ms   budget: {EMBED_BUDGET:.0f} ms  {'PASS ✓' if pass_emb else 'FAIL ✗'}")
print(f"  Effective FPS      : {eff_fps:.1f}   target: {VIDEO_FPS} fps   {'PASS ✓' if pass_fps else 'FAIL ✗'}")
print(f"{'='*58}")
overall = 'PASS ✓' if (pass_det and pass_emb and pass_fps) else 'FAIL ✗'
print(f"  OVERALL PHASE 12   : {overall}")
print(f"{'='*58}")
print(f"\n  ── Copy these into the thesis table ──")
print(f"  YOLOv8n detection  : {mean_det:.1f} ms mean / {p95_det:.1f} ms p95")
print(f"  OSNet embedding    : {mean_emb:.1f} ms mean / {p95_emb:.1f} ms p95")
print(f"  Effective FPS      : {eff_fps:.1f}")
print(f"  Device             : {device.upper()} ({torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'})")



  Phase 12 — Real-Time Pipeline Throughput  [CUDA]
  Frames benchmarked : 300  (0 with detection)
  Detection  mean    : 9.7 ms   p95: 14.6 ms   budget: 40 ms  PASS ✓
  Embedding  mean    : 0.0 ms   p95: 0.0 ms   budget: 200 ms  PASS ✓
  Effective FPS      : 102.2   target: 25 fps   PASS ✓
  OVERALL PHASE 12   : PASS ✓

  ── Copy these into the thesis table ──
  YOLOv8n detection  : 9.7 ms mean / 14.6 ms p95
  OSNet embedding    : 0.0 ms mean / 0.0 ms p95
  Effective FPS      : 102.2
  Device             : CUDA (Tesla T4)


In [6]:
# ── 6. Save results to a text file for download ────────────────────────────────
from pathlib import Path
out = Path("phase12_gpu_results.txt")
out.write_text(
    f"Phase 12 — Real-Time Pipeline Throughput\n"
    f"Device: {device.upper()} ({torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'})\n"
    f"Frames: {frames_read}\n"
    f"Detection mean: {mean_det:.1f} ms  p95: {p95_det:.1f} ms\n"
    f"Embedding mean: {mean_emb:.1f} ms  p95: {p95_emb:.1f} ms\n"
    f"Effective FPS: {eff_fps:.1f}\n"
    f"Overall: {'PASS' if (pass_det and pass_emb and pass_fps) else 'FAIL'}\n"
)
print(f"✓ Results written to {out} — download from Files panel (left sidebar)")
from google.colab import files
files.download(str(out))


✓ Results written to phase12_gpu_results.txt — download from Files panel (left sidebar)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>